In [ ]:
import nltk
from wcwidth import width

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
#Stage 1: Data Collection.

#import the review scraping function from library.
from google_play_scraper import reviews, Sort
import pandas as pd

#import time to add small pauses between requests so we don't hammer Google server.
import time

#define our apps as a dictionary where Key is the name as we'll use it and the value is the app's ID from Google Play Store.

apps = {
    'EcoCash'  : 'com.ecocash',
    'Innbucks' : 'com.innbucks.customer',
    'Omari'    : 'com.hypercubesoft.omari',
    'OneMoney' : 'zw.co.onemoney.mob',
    'Mukuru'   : 'com.mukuru.customer.android'
}

all_reviews = [] #master list.

#loop through each app, scraping.
for app_name, app_id in apps.items():
    print(f'Scraping reviews for: {app_name}...')
    result, _ = reviews(
        app_id,
        lang = 'en', #only english reviews
        country = 'zw', #only reviews from Zimbabwe
        sort = Sort.NEWEST, #get the latest reviews first
        count = 200, #pull 200 reviews per app as a start
    )

    #each result will be a dictionary, one per review. We'll loop through and add a label so we know which app each review came from.

    for review in result:
        review['app'] = app_name
        all_reviews.append(review)

    #2 second pause between requests.
    time.sleep(2)

print('Scraping done.')

#convert dictionaries into a DF.
df = pd.DataFrame(all_reviews)

print(df.shape)
print(df.head())

df.to_csv('data/raw_reviews.csv', index=False)
print("Raw data saved.")

#for now, we'll keep the columns that we need for sentiment analysis.

df = df[['app', 'userName', 'score', 'content', 'at']]
#rename for clarity and readability
df.columns = ['app', 'reviewer', 'rating', 'review_text', 'date']

df.to_csv('data/clean_reviews.csv', index=False)




In [ ]:
#Stage 2: Exploratory Data Analysis.
import matplotlib.pyplot as plt
import seaborn as sns
#1. Initial Audit.
df = pd.read_csv('data/clean_reviews.csv')
print('\nData Shape: ', df.shape)
print(df.head())
print('\nData Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

#Fix date column from str.
df['date'] = pd.to_datetime(df['date'])
print('\nData Types:')
print(df.dtypes)

#are reviews evenly spread across 1 app or dominated?
print(df['app'].value_counts())

#rating distribution
print('\nRating Distribution:')
print(df['rating'].value_counts().sort_index())

# apply same global style as temporal plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'axes.edgecolor'    : 'black',
    'axes.labelcolor'   : 'black',
    'xtick.color'       : 'black',
    'ytick.color'       : 'black',
    'text.color'        : 'black',
    'grid.color'        : '#cccccc',
    'grid.linewidth'    : 0.6,
    'legend.facecolor'  : 'white',
    'legend.edgecolor'  : 'black',
    'legend.labelcolor' : 'black',
})

# your custom app colour palette
app_colours = {
    'EcoCash'  : '#87CEEB',
    'Innbucks' : '#1B2A4A',
    'Omari'    : '#2E8B57',
    'OneMoney' : '#FF8C00',
    'Mukuru'   : '#DC143C'
}

# build ordered colour list matching the rating plot's app order
app_palette = list(app_colours.values())

#overall rating spread
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='rating', palette=app_palette)
plt.title('Overall rating distribution', color='black')
plt.xlabel('Rating (1=worst, 5=best)', color='black')
plt.ylabel('Count', color='black')
plt.tight_layout()
plt.savefig('plots/rating_distribution.png', dpi=150, facecolor='white')
plt.show()

#rating distribution - does sentiment differ across brands?
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='rating', hue='app', palette=app_colours)
plt.title('Rating distribution by app', color='black')
plt.xlabel('Rating', color='black')
plt.ylabel('Count', color='black')
plt.legend(title='App', facecolor='white', edgecolor='black', labelcolor='black')
plt.tight_layout()
plt.savefig('plots/rating_by_app.png', dpi=150, facecolor='white')
plt.show()

#review length (how many words are in each review? Longer words might carry richer sentiments.
df['review_length'] = df['review_text'].dropna().apply(lambda x: len(x.split()))
print('\nReview length statistics:')
print(df['review_length'].describe())

plt.figure(figsize=(8, 5))
sns.histplot(df['review_length'], bins=40, kde=True, color='#1B2A4A')
plt.title('Distribution of review length (words)', color='black')
plt.xlabel('Word count', color='black')
plt.ylabel('Frequency', color='black')
plt.tight_layout()
plt.savefig('plots/review_length.png', dpi=150, facecolor='white')
plt.show()

df.to_csv('data/eda_reviews.csv', index=False)
print("EDA dataset saved.")

In [ ]:
#Stage 3: NLP Preprocessing.
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

df = pd.read_csv('data/clean_reviews.csv')
df['date'] = pd.to_datetime(df['date'])
print(df.dtypes)

#initialise lemmatiser.
lemmatiser= WordNetLemmatizer()
#load standard english stopwords list
stop_words = set(stopwords.words('english'))
#exclude negation words from stopwords.
negations = {
    "not", "no", "nor", "never", "neither",
    "cannot", "cant", "cant", "won't", "wont",
    "don't", "dont", "didn't", "didnt",
    "doesn't", "doesnt", "isn't", "isnt",
    "wasn't", "wasnt", "weren't", "werent",
    "haven't", "havent", "hasn't", "hasnt",
    "hadn't", "hadnt", "wouldn't", "wouldnt",
    "shouldn't", "shouldnt", "couldn't", "couldnt"
}

#Track 2 Function: minimal cleaning for VADER & TextBlob sentiment scoring and transformer model.
#our model should understand 'can't login!!!' with the intended user frustration. VADER CAN, AND VADER WILL.
#subtract negations from stopwords, all words in the list will remain after stopword removal
stop_words = stop_words - negations
print(f'Stopwords loaded: {len(stop_words)} words')
print(f'Negations kept: {len(negations)} words')

#light cleaning (strip URLs and lowercase, but keep everything else intact)to preserve the richness that VADER needs.

def light_clean(text):
    text = text.lower() #lower everything.
    text = re.sub(r'http\S+|www\S+', '', text) #remove URL
    text = re.sub(r'@\w+', '', text) #remove usernames like @ecocash
    text = text.strip() #clear whitespace

    return text

#Track 1: Full 5 steps of NLP for word frequency counts, word clouds and identifying the most common themes per app.

def full_clean(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text) #URLs
    text = re.sub(r'@\w+', '', text) #usernames
    text = re.sub(r'[^a-z\s]', '', text) #punctuation and special characters
    text = re.sub(r'\s+', ' ', text).strip() #extra whiitespace
    tokens = word_tokenize(text) #Punkt magic: split text into individual word tokens
    tokens = [t for t in tokens if t not in stop_words] #filter out any token that appears in our stopwords. Negations surveive.
    tokens = [lemmatiser.lemmatize(t) for t in tokens]
    return ' '.join(tokens) #reduce tokens to base dictionary form

#Apply both tracks.
print('\nLight cleaning for sentiment track...') #track 2.
df['sentiment_text'] = df['review_text'].apply(light_clean)

print('\nFull NLP for analysis track...') #track 1.
df['clean_tokens'] = df['review_text'].apply(full_clean)

print('\nPreprocessing done.')

#Sanity check.
check = df[['review_text', 'sentiment_text', 'clean_tokens']].head(10)

for i, row in check.iterrows():
    print(f"\n--- Review {i+1} ---")
    print(f"ORIGINAL  : {row['review_text']}")
    print(f"SENTIMENT : {row['sentiment_text']}")
    print(f"TOKENS    : {row['clean_tokens']}")

#save our dataset.
df.to_csv('data/preprocessed_reviews.csv', index=False)

print(df.shape)
print(df.columns.tolist())


In [ ]:
#Stage 4: Sentiment Modelling.
    #1. VADER.
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

#load data
df = pd.read_csv('data/preprocessed_reviews.csv')
#initialise vader.
#NB: SentimentIntensityAnalyzer holds the sentiment dict. and all the rules.

sia = SentimentIntensityAnalyzer()


def get_vader_scores(text): #takes text and returns VADER's compound score.
    scores = sia.polarity_scores(text) #polarity scores return a dict. with 4 keys; pos, neg, neu and compund.
    return scores['compound']

#apply VADER to sentiment_text column ->TRACK 2

print('\nRunning VADER...')
df['vader_compound'] = df['sentiment_text'].apply(get_vader_scores)

#convert the compound scores to human-readable labels.
#>= 0.05 ->Positive, <=-0.05 -> Negative, inbetween -> Neutral.

def vader_label(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['vader_label'] = df['vader_compound'].apply(vader_label)

#sanity check.
print(df[['review_text', 'vader_compound', 'vader_label']].head(10))
print('\nVader label distribution:')
print(df['vader_label'].value_counts())

    #TextBlob
from textblob import TextBlob
def get_textblob_scores(text):

    analysis = TextBlob(text)
    #polarity: -1(-ve) to +1(+ve)| subjectiveity: 0(factual) to 1(opinion)

    return pd.Series({
        'textblob_polarity': analysis.sentiment.polarity,
        'textblob_subjectivity': analysis.sentiment.subjectivity
    })

print('\nRunning TextBlob...')
df[['textblob_polarity', 'textblob_subjectivity']] = df['sentiment_text'].apply(get_textblob_scores)

#convert textblob polarity to a label, consistent with VADER.
def textblob_label(score):
    if score > 0.05:
        return 'Positive'
    elif score < -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['textblob_label'] = df['textblob_polarity'].apply(textblob_label)

print(df[['review_text', 'textblob_polarity', 'textblob_subjectivity', 'textblob_label']].head(10))
print('\nTextBlob label distribution:')
print(df['textblob_label'].value_counts())

    #Transformer model
from transformers import pipeline
print('Loading Transformer model -- this may take a while...')

sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model='cardiffnlp/twitter-roberta-base-sentiment'
)

#NB: Transformers have a token limit, they can only read a certain number of characters at once so we'll limit each review to 512 characters. Our data has a nediam of 5 words, so we should be OK.

def get_transformer_label(text):
    result = sentiment_pipeline(text[:512])[0]['label']

    #map model labels to ours.
    label_map = {
        'LABEL_0' : 'Negative',
        'LABEL_1' : 'Neutral',
        'LABEL_2' : 'Positive'
    }
    return label_map[result]

print("Running transformer model -- this will take longer than VADER and TextBlob...")
df['transformer_label'] = df['sentiment_text'].apply(get_transformer_label)

print(df[['review_text', 'transformer_label']].head(10))
print("\nTransformer label distribution:")
print(df['transformer_label'].value_counts())

    #MODEL COMPARISON
#We want to compare how often all 3 models agree on the same label. While perfect agreement in the models will signal high confidence in the review's sentiment, Disagreement will signal genuine ambiguity in the review.

def agreement(row):
    labels = [
        row['vader_label'],
        row['textblob_label'],
        row['transformer_label']
    ]
    #all 3 match = full agreement.
    if labels[0] == labels[1] == labels[2]:
        return 'All agree'
    #if 2 match = partial agreement
    elif labels[0] == labels[1] or labels[1] == labels[2] or labels[0] == labels[2]:
        return 'Two agree'
    else:
        return 'All disagree'

df['model_agreement'] = df.apply(agreement, axis=1)

print('\nModel agreement distribution:')
print(df['model_agreement'].value_counts())

#let's take a gander at reviews where all 3 models disagree
print("\nReviews where all models disagree:")
print(df[df['model_agreement'] == 'All disagree'][['review_text', 'vader_label', 'textblob_label', 'transformer_label']].head(10))

df.to_csv('data/scored_reviews.csv', index=False)
print("\nScored dataset saved.")

#final check
print(df.shape)
print(df.columns.tolist())



In [ ]:
#Stage 5: Temporal Analysis.

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

df = pd.read_csv('data/scored_reviews.csv')

df['date'] = pd.to_datetime(df['date'])

#set a consistent colour palette for this stage.
app_colours = {
    'EcoCash' : '#87CEEB',
    'Innbucks' : '#1B2A4A',
    'Omari': '#2E8B57',
    'OneMoney' : '#FF8C00',
    'Mukuru' : '#DC143C'
}

print('\nData loaded.')
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(df['app'].value_counts())

   # Set global plot aesthetics -- applied to all visualisations below
# white background, black text, clean gridlines
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor'  : 'white',      # outer figure background
    'axes.facecolor'    : 'white',      # plot area background
    'axes.edgecolor'    : 'black',      # axis border
    'axes.labelcolor'   : 'black',      # x and y axis label text
    'xtick.color'       : 'black',      # x tick labels
    'ytick.color'       : 'black',      # y tick labels
    'text.color'        : 'black',      # all other text
    'grid.color'        : '#cccccc',    # light grey gridlines
    'grid.linewidth'    : 0.6,
    'legend.facecolor'  : 'white',      # legend background
    'legend.edgecolor'  : 'black',      # legend border
    'legend.labelcolor' : 'black',      # legend text
})

#1st visual: Monthly sentiment trend.
#resample data to monthly freq. per app.
monthly_sentiment = (
    df.groupby(['app', pd.Grouper(key='date', freq='ME')])['vader_compound'].mean().reset_index()
)

#rename columns for clarity
monthly_sentiment.columns = ['app', 'month', 'avg_sentiment']

print('\nMonthly sentiment sample:')
print(monthly_sentiment.head(10))

fig, ax = plt.subplots(figsize=(12, 6))
for app in monthly_sentiment['app'].unique():
    app_data = monthly_sentiment[monthly_sentiment['app'] == app] #filter to 1 app at a time.

    ax.plot(
        app_data['month'],
        app_data['avg_sentiment'],
        label=app,
        color=app_colours[app],
        linewidth=2,
        marker='o',
        markersize=4,
    )

#add a straight line at zero, separate positive and negative reviews
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5, label='Neutral boundary')

ax.set_title('Monthly sentiment trend by app (VADER)', fontsize=14, color='black')
ax.set_xlabel('Month', color='black')
ax.set_ylabel('Average compound sentiment score', color='black')
ax.legend(title='App', bbox_to_anchor=(1.05, 1), loc='upper left')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('plots/monthly_sentiment_trend.png', dpi=150, facecolor='white')
plt.show()
print("Visualisation 1 saved.")

    #2nd visual: Rolling sentiment average
#one bad month can skew everything, a rolling average can smoothe this by averaging each point.

fig, ax = plt.subplots(figsize=(12, 6))

for app in monthly_sentiment['app'].unique():
    app_data = monthly_sentiment[monthly_sentiment['app'] == app]
    app_data = app_data.sort_values('month')
    app_data['rolling_avg'] = app_data['avg_sentiment'].rolling(
        window = 2,
        min_periods = 1,
    ).mean()

    ax.plot(
        app_data['month'],
        app_data['avg_sentiment'],
        color = app_colours[app],
        alpha = 0.25,
        linewidth = 1,
    )

    #plot the rolling average
    ax.plot(
        app_data['month'],
        app_data['rolling_avg'],
        label=app,
        color=app_colours[app],
        linewidth=2.5,
    )
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5, label='Neutral boundary')
ax.set_title('Rolling average sentiment trend by app (2-month window)', fontsize=14, color='black')
ax.set_xlabel('Month', color='black')
ax.set_ylabel('Rolling average compound score', color='black')
ax.legend(title='App', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('plots/rolling_sentiment_trend.png', dpi=150, facecolor='white')
plt.show()
print("Visualisation 2 saved.")

    #3rd visual: Review volume over time.
monthly_volume = (
    df.groupby(['app', pd.Grouper(key='date', freq='ME')])
    .size()
    .reset_index()
)
monthly_volume.columns = ['app', 'month', 'review_count']

fig, ax = plt.subplots(figsize=(12, 6))

for app in monthly_volume['app'].unique():
    app_data = monthly_volume[monthly_volume['app'] == app]
    ax.plot(
        app_data['month'],
        app_data['review_count'],
        label     = app,
        color     = app_colours[app],
        linewidth = 2,
        marker    = 'o',
        markersize = 4
    )

ax.set_title('Monthly review volume by app', fontsize=14, color='black')
ax.set_xlabel('Month', color='black')
ax.set_ylabel('Number of reviews', color='black')
ax.legend(title='App', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('plots/review_volume.png', dpi=150, facecolor='white')
plt.show()
print("Visualisation 3 saved.")

temporal_summary = pd.merge(monthly_sentiment, monthly_volume, on=['app', 'month'])
temporal_summary.to_csv('data/temporal_summary.csv', index=False)
print("\nTemporal summary saved.")

In [ ]:
#Stage 6: Visualisation and Reporting.
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from wordcloud import WordCloud
import numpy as np

df = pd.read_csv('data/scored_reviews.csv')
df['date'] = pd.to_datetime(df['date'])

#match visualisation style.
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'axes.edgecolor'    : 'black',
    'axes.labelcolor'   : 'black',
    'xtick.color'       : 'black',
    'ytick.color'       : 'black',
    'text.color'        : 'black',
    'grid.color'        : '#cccccc',
    'grid.linewidth'    : 0.6,
    'legend.facecolor'  : 'white',
    'legend.edgecolor'  : 'black',
    'legend.labelcolor' : 'black',
})

app_colours = {
    'EcoCash'  : '#87CEEB',
    'Innbucks' : '#1B2A4A',
    'Omari'    : '#2E8B57',
    'OneMoney' : '#FF8C00',
    'Mukuru'   : '#DC143C'
}

sentiment_colours = {
    'Positive' : '#2E8B57',
    'Neutral'  : '#FF8C00',
    'Negative' : '#DC143C'
}

    #1st Visual: Sentiment distribution by app.
#what porportion of reviews for each app were positive, negative and neutral across all 3 models?

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey = True)
models = [
    ('vader_label', 'VADER'),
    ('textblob_label', 'TextBlob'),
    ('transformer_label', 'Transformer')
]

for ax, (col, title) in zip(axes, models):
    proportions = (
        df.groupby([ 'app', col]).size().groupby(level=0).apply(lambda x: x / x.sum()*100).unstack(fill_value=0)
    )

    for label in ['Positive', 'Neutral', 'Negative']:
        if label not in proportions.columns:
            proportions[label] = 0

    proportions = proportions[['Positive', 'Neutral', 'Negative']]

    proportions.plot(
        kind = 'bar',
        ax = ax,
        color = [sentiment_colours[l] for l in ['Positive', 'Neutral', 'Negative']],
        width = 0.75,
        edgecolor = 'white',
    )

    ax.set_title(title, fontsize=13, color='black')
    ax.set_xlabel('App', color='black')
    ax.set_ylabel('Percentage of reviews (%)', color='black')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Sentiment', facecolor='white', edgecolor='black')

fig.suptitle('Sentiment distribution by app across all three models', fontsize=15, color='black')
plt.tight_layout()
plt.savefig('plots/sentiment_distribution_by_app.png', dpi=150, facecolor='white')
plt.show()
print("Visualisation 1 saved.")

    #2nd Visual: Subjectivity vs. Polarity.
#TextBlob gave dimensions for polarity (-1 to +1) and subjectivity (0 to 1). Plotting these against each other will reveal cluster patterns.

fig, ax = plt.subplots(figsize=(10, 7))

for app, colour in app_colours.items():
    app_data = df[df['app'] == app]
    ax.scatter(
        app_data['textblob_polarity'], app_data['textblob_subjectivity'], label=app, color=colour, alpha=0.5, edgecolor='white', linewidth=0.3, s=40
    )

#divide charts into 4 quadrants, (Vertical line at 0 separates +ve and -ve polarity | horizontal line at 0.5 separates facts and opinions.
ax.axvline(x=0,   color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(y=0.5, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

#quadrant labels.
ax.text( 0.85, 0.97, 'Opinionated\npositive',  transform=ax.transAxes, fontsize=9, color='black', va='top', ha='right')
ax.text( 0.15, 0.97, 'Opinionated\nnegative',  transform=ax.transAxes, fontsize=9, color='black', va='top', ha='left')
ax.text( 0.85, 0.03, 'Factual\npositive',       transform=ax.transAxes, fontsize=9, color='black', va='bottom', ha='right')
ax.text( 0.15, 0.03, 'Factual\nnegative',       transform=ax.transAxes, fontsize=9, color='black', va='bottom', ha='left')

ax.set_title('Subjectivity vs polarity by app (TextBlob)', fontsize=14, color='black')
ax.set_xlabel('Polarity (negative ← 0 → positive)', color='black')
ax.set_ylabel('Subjectivity (factual ← 0.5 → opinionated)', color='black')
ax.legend(title='App', facecolor='white', edgecolor='black')

plt.tight_layout()
plt.savefig('plots/subjectivity_vs_polarity.png', dpi=150, facecolor='white')
plt.show()
print("Visualisation 2 saved.")

    #3rd Visual: Word clouds per app.
#built from clean_tokens column in Track 1, the more frequent the word appears, the larger it'll appear, making it easier for dominant themes to stand out.

#exclude dominant words that tell us nothing, including app names.
excluded_words = {
    'app', 'application',
    'ecocash', 'innbucks', 'omari', 'onemoney', 'mukuru',
    'one', 'money', 'device'
}

fig = plt.figure(figsize=(18, 10))

for i, (app, colour) in enumerate(app_colours.items()):

    #combine all tokens for this app into 1 long string, wordcloud takes strings, not lists.
    app_tokens = ' '.join(
        df[df['app'] == app]['clean_tokens'].dropna().tolist()
    )

    #generate word cloud.
    wc = WordCloud(
        width            = 800,
        height           = 400,
        background_color = 'white',
        max_words        = 80,   #limit clutter, top 80 is plenty
        colormap         = 'Blues' if colour == '#87CEEB' else 'Greens' if colour == '#2E8B57' else 'Oranges' if colour == '#FF8C00' else 'Reds' if colour == '#DC143C' else 'Blues',
        collocations     = False, #prevents the same 2-word phrase dominating
        stopwords        = excluded_words  # filters out our custom exclusion list
    ).generate(app_tokens)

    #add subplots
    ax = fig.add_subplot(2, 3, i + 1)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(app, fontsize=13, color='black', fontweight='bold')
    ax.axis('off')   # word clouds don't need axes

fig.suptitle('Most frequent words in reviews by app', fontsize=15, color='black')
plt.tight_layout()
plt.savefig('plots/wordclouds_by_app.png', dpi=150, facecolor='white')
plt.show()
print("Visualisation 3 saved.")



    #4th Visual: Sentiment heatmap.
#show average VADER sentiment by app and by month, Rows = apps, Columns = months, Colour = average sentiment score

#pivot table.
monthly_sentiment = (
    df.groupby(['app', pd.Grouper(key='date', freq='ME')])['vader_compound']
    .mean()
    .reset_index()
)
monthly_sentiment.columns = ['app', 'month', 'avg_sentiment']

monthly_sentiment['month_label'] = monthly_sentiment['month'].dt.strftime('%b %Y')

#pivot into matrix format for sns heatmap
heatmap_data = monthly_sentiment.pivot(
    index   = 'app',
    columns = 'month_label',
    values  = 'avg_sentiment'
)

#reorder columns.
month_order = (
    monthly_sentiment[['month', 'month_label']]
    .drop_duplicates()
    .sort_values('month')['month_label']
    .tolist()
)
heatmap_data = heatmap_data[month_order]

fig, ax = plt.subplots(figsize=(16, 5))

sns.heatmap(
    heatmap_data,
    ax         = ax,
    cmap       = 'RdYlGn',    # red = negative, yellow = neutral, green = positive
    center     = 0,            # centres the colour scale at zero (neutral)
    annot      = True,         # shows the actual score in each cell
    fmt        = '.2f',        # two decimal places
    linewidths = 0.5,          # thin lines between cells for readability
    linecolor  = 'white',
    cbar_kws   = {'label': 'Average VADER compound score'}
)

ax.set_title('Sentiment heatmap — average VADER score by app and month', fontsize=14, color='black')
ax.set_xlabel('Month', color='black')
ax.set_ylabel('App', color='black')
plt.xticks(rotation=45, ha='right', color='black')
plt.yticks(rotation=0, color='black')

plt.tight_layout()
plt.savefig('plots/sentiment_heatmap.png', dpi=150, facecolor='white')
plt.show()
print("Visualisation 4 saved.")